# Mining Frequent Patterns — In-Class Practice

**CSCI 480 / CIS 680 — Advanced Databases**

Adapted from FAU-CS6, *KDD — Knowledge Discovery in Databases*, Exercise 1: Mining Frequent Patterns (Tasks 1–3). The transaction data have been aligned with the lecture slides.

**Learning goals:** encode transaction data, identify frequent itemsets, generate association rules, and interpret support, confidence, and lift. Estimated time: Part 1 about 25–35 minutes; Part 2 requires additional time.

## Setup
Run the cells in order. This notebook needs `pandas` and `mlxtend`. If `mlxtend` is missing, run the installation cell once, then rerun the imports. Internet access may be required for installation.

In [ ]:
# Uncomment and run if necessary:
# %pip install pandas mlxtend

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules


## Transaction data
Each set represents the products purchased in one transaction. **These five baskets match the FAU lecture example**, including Milk in transaction 5.

| TID | Items |
|---|---|
| 10 | Beer, Nuts, Diapers |
| 20 | Beer, Coffee, Diapers |
| 30 | Beer, Diapers, Eggs |
| 40 | Nuts, Eggs, Milk |
| 50 | Nuts, Coffee, Diapers, Eggs, Milk |

In [ ]:
dataset = [
    {"Beer", "Nuts", "Diapers"},
    {"Beer", "Coffee", "Diapers"},
    {"Beer", "Diapers", "Eggs"},
    {"Nuts", "Eggs", "Milk"},
    {"Nuts", "Coffee", "Diapers", "Eggs", "Milk"},
]
print("Transactions:", len(dataset))
dataset

## Task 1 — Prepare the transactions
`fpgrowth()` expects a one-hot-encoded pandas DataFrame: one row per transaction, one Boolean column per product.

Use `TransactionEncoder` to encode `dataset`, then construct a DataFrame named `preprocessed_dataset`. Display the result.

**Hint:** `TransactionEncoder().fit(dataset).transform(dataset)` produces the Boolean array; use `encoder.columns_` for the column names.


In [ ]:
# TODO Task 1: build preprocessed_dataset



## Task 2 — Find frequent itemsets
Find all frequent itemsets using **minimum relative support = 0.4** (at least 2 of the 5 transactions). Store the result in `frequent_itemsets` and display it.

Use `fpgrowth()` with `use_colnames=True`. You do not need to study how the algorithm works internally.

**Check your reasoning:** How many transactions contain `{Beer, Diapers}`? What is its relative support? Which itemsets meet a stricter threshold of 0.6?


In [ ]:
# TODO Task 2: calculate frequent_itemsets



## Task 3 — Generate association rules
Use `association_rules()` to generate rules with **minimum support = 0.4**, using the frequent itemsets from Task 2. Store them in `frequent_patterns` and display at least `antecedents`, `consequents`, `support`, `confidence`, and `lift`.

**Hint:** `metric="support"` and `min_threshold=0.4`. Depending on the installed `mlxtend` version, `association_rules` may also require `num_itemsets=len(dataset)`.

In [ ]:
# TODO Task 3: generate frequent_patterns



## Part 2 — Mining transaction data from AdventureWorks (SQLite)

**Adapted from FAU-CS6/KDD, Mining Frequent Patterns, Task 4.** You are analyzing which AdventureWorks products are purchased together. Here SQL retrieves the relational data, while Python performs the frequent-pattern mining.

**Codespaces setup:** The real AdventureWorks SQLite database (~123 MB) is installed automatically when a new Codespace is created. You do not need to download it manually. If the database is missing or invalid, run `bash .devcontainer/setup.sh` in the Codespaces terminal and then rerun the connection cell.


In [ ]:
from pathlib import Path
import sqlite3

database_path = Path("data/adventure-works.db")
if not database_path.is_file():
    raise FileNotFoundError(
        "AdventureWorks database is missing. Codespaces should install it automatically. "
        "Open Terminal and run: bash .devcontainer/setup.sh"
    )
with database_path.open("rb") as database_file:
    if database_file.read(16) != b"SQLite format 3\x00":
        raise RuntimeError(
            "AdventureWorks file is not a valid SQLite database (possibly a Git LFS pointer). "
            "Open Terminal and run: bash .devcontainer/setup.sh"
        )
connection = sqlite3.connect(database_path)
print(f"Connected to AdventureWorks SQLite: {database_path} ({database_path.stat().st_size / (1024*1024):.1f} MiB)")


### Task 4 — Explore the relational tables

Run the queries below. Examine `TransactionHistory` and `Product`. Which columns identify a transaction record, an order, and a product?

In [ ]:
transaction_history_df = pd.read_sql_query(
    "SELECT * FROM TransactionHistory;", connection
)
product_df = pd.read_sql_query(
    "SELECT * FROM Product;", connection
)

print("TransactionHistory:", transaction_history_df.shape)
display(transaction_history_df.head(10))
print("Product:", product_df.shape)
display(product_df[["ProductID", "Name"]].head(10))

### Task 5 — Determine what represents a shopping basket

FAU highlights a database-modeling issue: `TransactionID` identifies a row, but `ReferenceOrderID` can group products purchased together. Use SQL to test this.

**Question:** Why is grouping by `TransactionID` inappropriate for association-rule mining?

In [ ]:
# Run these two SQL queries and compare the results.
by_transaction = pd.read_sql_query("""
SELECT TransactionID, COUNT(DISTINCT ProductID) AS product_count
FROM TransactionHistory
GROUP BY TransactionID
HAVING COUNT(DISTINCT ProductID) > 1
LIMIT 10;
""", connection)

by_order = pd.read_sql_query("""
SELECT ReferenceOrderID, COUNT(DISTINCT ProductID) AS product_count
FROM TransactionHistory
GROUP BY ReferenceOrderID
HAVING COUNT(DISTINCT ProductID) > 1
LIMIT 10;
""", connection)

print("TransactionIDs with multiple products:")
display(by_transaction)
print("ReferenceOrderIDs with multiple products:")
display(by_order)

### Task 6 — Prepare shopping baskets

Use `ReferenceOrderID` to group `ProductID` values into baskets. Convert the baskets into a Boolean table with `TransactionEncoder`.

**Student task:** Complete the code below. The one-hot-encoded table should have one row per order and one column per product.

In [ ]:
# TODO: group the transaction_history_df by ReferenceOrderID.
# Hint: use .groupby("ReferenceOrderID")["ProductID"].apply(lambda x: list(set(x)))
products_per_order = None

# TODO: use TransactionEncoder to convert products_per_order to a Boolean DataFrame.
# Store the result in adventure_baskets.
adventure_baskets = None

### Task 7 — Discover and interpret association rules

Use `fpgrowth()` to discover frequent product sets and `association_rules()` to generate rules. Following the FAU exercise, try minimum itemset support **0.01** and minimum rule support **0.02**. Select the ten rules with the highest support.

**Student task:** Complete the code. Then translate product IDs into names using the `Product` table.

*Note:* FAU uses support as a practical interpretation of the ambiguous word “relevant”; other rankings, such as confidence or lift, answer different questions.

In [ ]:
# TODO: find frequent itemsets from adventure_baskets using fpgrowth.
# TODO: generate association rules with association_rules.
# TODO: select the ten highest-support rules.
# TODO: map ProductID values in antecedents and consequents to product names.
# Hint: product_df.set_index("ProductID")["Name"] creates a lookup.

# Your code here

### Reflection

1. Why must relational rows be grouped into orders before mining?
2. What is the difference between `TransactionID` and `ReferenceOrderID`?
3. Why are product names more useful than numeric product IDs in a report?
4. Does high support necessarily imply a high lift? Explain.

**Source:** FAU-CS6/KDD, *Mining Frequent Patterns*, Task 4, and [FAU-CS6/KDD-Databases](https://github.com/FAU-CS6/KDD-Databases). This classroom version uses FP-growth only and omits Apriori and the advanced interestingness-measure tasks.

In [ ]:
# Optional: close the connection after completing Part 2.
# connection.close()